In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import PromptTemplate, MessagesPlaceholder

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory

from langchain_openai import ChatOpenAI

In [10]:
# Point to your local MLX server
llm = ChatOpenAI(
    base_url="http://localhost:8080/v1",
    api_key="not-needed",
    # Change "local-model" to the exact name shown in your server logs
    model="mlx-community/Qwen2.5-1.5B-Instruct-4bit", 
    temperature=0.7
)


In [4]:
parser = StrOutputParser()

In [5]:
template = "create a title for the story about {summary}. ONLY RETURN THE TITLE."
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = RunnablePassthrough.assign(title=title_prompt | llm | parser)

template = "Describe the main character of the story {summary} with the title {title}. Use only two sentences"
character_prompt = PromptTemplate(template=template, input_variables=["summary", "title"])
character = RunnablePassthrough.assign(character=character_prompt | llm | parser)

template = "Create a story about {summary} with the title {title}. The main character description is {character}. Only return one paragraph."
story_prompt = PromptTemplate(template=template, input_variables=["summary", "title", "character"])
story = story_prompt | llm | parser

full_chain = title | character | story


full_chain.invoke({"summary": "a father lost her daughter"})

'In "Unseen Threads: A Father\'s Solo Journey," a father navigates the dark of night to reclaim his lost daughter\'s life, encountering unseen threads that weave through his solitude and hope. The threads, though unseen, are a symbol of his relentless pursuit and his profound connection to his daughter\'s spirit that remains invisible but ever-present. As he spirals deeper into his search, he finds solace in the support of his community, guided by unseen figures who offer comfort and hope, despite the challenges they face. In this journey, he learns that the unseen threads are a testament to the connections we make in life, even when they are not visible to the naked eye.'